# Mini EDA for IMOEX options dataset

Цель: понять структуру датасета, покрытие по датам, страйкам и ценам, а также оценить, насколько данные подходят для дальнейшего анализа.

Источник: `data/imoex_options_2024_2026/imoex_options_daily_history_2024_2026.parquet`


In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

data_path = Path(r'/Users/maria/Desktop/Code/HSE/COURSEBOOK/data/imoex_options_2024_2026/imoex_options_daily_history_2024_2026.parquet')
df = pd.read_parquet(data_path)
df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'], errors='coerce')
df['expiry_date'] = pd.to_datetime(df['expiry_date'], errors='coerce')
df['prefix'] = df['SECID'].astype(str).str[:2]
df.head()


In [ ]:
summary = pd.DataFrame({
    'metric': [
        'rows', 'columns', 'unique_secids', 'date_min', 'date_max',
        'unique_strikes', 'expiry_min', 'expiry_max'
    ],
    'value': [
        len(df),
        df.shape[1],
        df['SECID'].nunique(),
        df['TRADEDATE'].min(),
        df['TRADEDATE'].max(),
        df['strike'].nunique(),
        df['expiry_date'].min(),
        df['expiry_date'].max(),
    ]
})
summary


In [ ]:
print('Rows by year:')
display(df['TRADEDATE'].dt.year.value_counts().sort_index().to_frame('rows'))

print('Rows by month:')
display(df['TRADEDATE'].dt.to_period('M').astype(str).value_counts().sort_index().to_frame('rows'))

print('Unique SECIDs by year:')
display(df.groupby(df['TRADEDATE'].dt.year)['SECID'].nunique().to_frame('unique_secids'))


In [ ]:
print('Option type distribution:')
display(df['option_type'].value_counts(dropna=False).to_frame('rows'))

print('Strike summary for the whole dataset:')
display(df['strike'].describe().to_frame('value'))

print('Top strikes by row count:')
display(df['strike'].value_counts().head(20).to_frame('rows'))


## Проверка масштаба страйков

Подозрение на неверный масштаб разумное: в данных есть страйки порядка `2,500-3,500` и одновременно `165,000-380,000`. Сначала проверим, не смешаны ли здесь разные семейства контрактов.


In [ ]:
print('SECID prefix counts:')
display(df['prefix'].value_counts().to_frame('rows'))

print('Strike distribution by prefix:')
display(df.groupby('prefix')['strike'].describe()[['count', 'min', '25%', '50%', '75%', 'max']])

print('Sample rows with high strikes:')
display(df[df['strike'] >= 100000][['SECID', 'TRADEDATE', 'strike', 'SETTLEPRICE', 'option_type', 'expiry_date']].head(20))

print('Sample rows with low strikes:')
display(df[df['strike'] < 100000][['SECID', 'TRADEDATE', 'strike', 'SETTLEPRICE', 'option_type', 'expiry_date']].head(20))


In [ ]:
df['settle_to_strike'] = pd.to_numeric(df['SETTLEPRICE'], errors='coerce') / pd.to_numeric(df['strike'], errors='coerce')
print('SETTLEPRICE / strike by prefix:')
display(df.groupby('prefix')['settle_to_strike'].describe()[['count', 'min', '25%', '50%', '75%', 'max']])

print('Rows with settle/strike > 0.9:')
display(df[df['settle_to_strike'] > 0.9][['SECID', 'TRADEDATE', 'strike', 'SETTLEPRICE', 'settle_to_strike', 'option_type']].head(30))


### Вывод по масштабу

Похоже, это **не просто ошибка масштаба в одном столбце**, а смесь разных семейств опционных контрактов:
- `IM` и `MM` живут в диапазоне страйков около `1,650-3,450`;
- `MX` живёт в диапазоне порядка `165,000-380,000`.

То есть pooled-анализ по всем страйкам сразу делать нельзя. Для анализа страйков и moneyness данные нужно рассматривать **по семействам контрактов отдельно**.


## Проверка покрытия цен

Если для анализа нужен именно **ценовой ряд опциона**, то главное поле здесь — `SETTLEPRICE`. Проверим, насколько оно покрыто, и насколько покрыты торговые поля (`OPEN`, `CLOSE`, `VALUE`, `VOLUME`).


In [ ]:
missing_share = (df.isna().mean().sort_values(ascending=False) * 100).round(2).to_frame('missing_pct')
print('Missing share by column (%):')
display(missing_share)

secid_stats = df.groupby('SECID').agg(
    rows=('TRADEDATE', 'size'),
    settle_non_null=('SETTLEPRICE', lambda s: s.notna().sum()),
    open_non_null=('OPEN', lambda s: s.notna().sum()),
    close_non_null=('CLOSE', lambda s: s.notna().sum()),
    value_non_null=('VALUE', lambda s: s.notna().sum()),
    volume_non_null=('VOLUME', lambda s: s.notna().sum()),
)
secid_stats['settle_cov'] = secid_stats['settle_non_null'] / secid_stats['rows']
secid_stats['open_cov'] = secid_stats['open_non_null'] / secid_stats['rows']
secid_stats['close_cov'] = secid_stats['close_non_null'] / secid_stats['rows']

print('Coverage summary across SECIDs:')
display(secid_stats[['settle_cov', 'open_cov', 'close_cov']].describe())

print('How many SECIDs have full SETTLEPRICE coverage?')
print(int((secid_stats['settle_cov'] == 1).sum()), 'of', len(secid_stats))

print('How many SECIDs have any OPEN/CLOSE coverage?')
print('OPEN > 0 coverage SECIDs:', int((secid_stats['open_non_null'] > 0).sum()))
print('CLOSE > 0 coverage SECIDs:', int((secid_stats['close_non_null'] > 0).sum()))

print('SECIDs with no trade fields at all:')
print(int(((secid_stats['open_non_null'] == 0) & (secid_stats['close_non_null'] == 0) & (secid_stats['value_non_null'] == 0) & (secid_stats['volume_non_null'] == 0)).sum()))


In [ ]:
print('Best trade coverage SECIDs:')
display(secid_stats.sort_values(['open_cov', 'rows'], ascending=[False, False]).head(20))

print('Worst trade coverage SECIDs:')
display(secid_stats.sort_values(['open_cov', 'rows'], ascending=[True, True]).head(20))

print('Coverage by contract family (prefix):')
display(df.groupby('prefix').agg(
    rows=('SECID', 'size'),
    secids=('SECID', 'nunique'),
    open_cov=('OPEN', lambda s: s.notna().mean()),
    close_cov=('CLOSE', lambda s: s.notna().mean()),
    value_cov=('VALUE', lambda s: s.notna().mean()),
    volume_cov=('VOLUME', lambda s: s.notna().mean()),
    openpos_cov=('OPENPOSITION', lambda s: s.notna().mean()),
    settle_cov=('SETTLEPRICE', lambda s: s.notna().mean()),
))


## Насколько густая сетка страйков

Для ATM-анализа важно понять, сколько страйков вообще доступно на одну дату/экспирацию.


In [ ]:
grid_stats = (df.groupby(['TRADEDATE', 'expiry_date']).agg(
    rows=('SECID', 'size'),
    secids=('SECID', 'nunique'),
    strikes=('strike', 'nunique'),
    calls=('option_type', lambda s: (s == 'C').sum()),
    puts=('option_type', lambda s: (s == 'P').sum()),
    nonnull_open=('OPEN', lambda s: s.notna().sum()),
    nonnull_settle=('SETTLEPRICE', lambda s: s.notna().sum()),
).reset_index())

print('Strikes per TRADEDATE x expiry_date summary:')
display(grid_stats['strikes'].describe().to_frame('value'))

print('Worst date-expiry combinations:')
display(grid_stats.sort_values(['strikes', 'rows']).head(20))

print('Best date-expiry combinations:')
display(grid_stats.sort_values(['strikes', 'rows'], ascending=[False, False]).head(20))


## Итоговая оценка

### Что хорошо
- Датасет большой: десятки тысяч строк и сотни SECID.
- Есть покрытие по 2024, 2025 и части 2026.
- `SETTLEPRICE` заполнен для всех строк, то есть у нас действительно есть ценовой ряд.
- `option_type`, `strike`, `expiry_date` распарсены и выглядят полезными.

### Что надо учитывать
- Страйки нельзя анализировать все вместе: в данных смешаны разные семейства контрактов (`IM`, `MM`, `MX`) с разным масштабом.
- Торговые поля (`OPEN/HIGH/LOW/CLOSE/VALUE/VOLUME`) заметно грязнее, чем `SETTLEPRICE`.
- Есть контракты с почти чисто settlement-историей и без заметной торговой активности.
- На многих date-expiry комбинациях сетка страйков редкая, хотя местами она уже довольно плотная.

### Подходят ли данные нам?
- Да, **если основная цена опциона для анализа — это `SETTLEPRICE`**.
- Да, **если мы готовы анализировать данные по семействам контрактов отдельно**.
- С осторожностью, если нам нужны именно рыночные trade prices (`OPEN/CLOSE`) каждый день.
- Для ATM-анализа данные потенциально подходят, но тогда нужно дополнительно присоединить ряд underlying и строить ATM внутри каждой даты/экспирации/семейства.


## Какое семейство оставляем: IM, MM или MX

Теперь сравним семейства именно под нашу задачу: **получить опционы на фьючерс на индекс IMOEX и их цены в максимально полном виде**.


In [ ]:
meta_path = Path("/Users/maria/Desktop/Code/HSE/COURSEBOOK/data/imoex_options_2024_2026/validated_historical_secids.parquet")
meta = pd.read_parquet(meta_path)
meta["family"] = meta["secid"].astype(str).str[:2]

family_examples = {}
for fam in ["IM", "MM", "MX"]:
    cols = [c for c in ["secid", "shortname", "name", "type", "group", "strike", "expiry_date"] if c in meta.columns]
    family_examples[fam] = meta.loc[meta["family"] == fam, cols].head(8)

for fam in ["IM", "MM", "MX"]:
    print(f"\nFAMILY {fam}")
    display(family_examples[fam])

family_stats = []
for fam, sub in df.groupby("prefix"):
    secid_stats = sub.groupby("SECID").agg(
        rows=("TRADEDATE", "size"),
        open_cov=("OPEN", lambda s: s.notna().mean()),
        close_cov=("CLOSE", lambda s: s.notna().mean()),
        value_cov=("VALUE", lambda s: s.notna().mean()),
        volume_cov=("VOLUME", lambda s: s.notna().mean()),
        settle_cov=("SETTLEPRICE", lambda s: s.notna().mean()),
    )
    grid = sub.groupby(["TRADEDATE", "expiry_date"]).agg(strikes=("strike", "nunique")).reset_index()
    family_stats.append({
        "family": fam,
        "rows": len(sub),
        "unique_secids": sub["SECID"].nunique(),
        "date_min": sub["TRADEDATE"].min(),
        "date_max": sub["TRADEDATE"].max(),
        "unique_expiry": sub["expiry_date"].nunique(),
        "unique_strikes": sub["strike"].nunique(),
        "median_rows_per_secid": secid_stats["rows"].median(),
        "median_open_cov_per_secid": secid_stats["open_cov"].median(),
        "median_close_cov_per_secid": secid_stats["close_cov"].median(),
        "median_value_cov_per_secid": secid_stats["value_cov"].median(),
        "median_volume_cov_per_secid": secid_stats["volume_cov"].median(),
        "settle_cov_all_rows": sub["SETTLEPRICE"].notna().mean(),
        "mean_strikes_per_date_expiry": grid["strikes"].mean(),
        "median_strikes_per_date_expiry": grid["strikes"].median(),
        "max_strikes_per_date_expiry": grid["strikes"].max(),
    })

family_stats = pd.DataFrame(family_stats).sort_values("family")
display(family_stats)

print("Rows by year per family:")
display(pd.crosstab(df["prefix"], df["TRADEDATE"].dt.year))

atm_proxy = []
for fam, sub in df.groupby("prefix"):
    grp = sub.groupby(["TRADEDATE", "expiry_date"])
    med = grp["strike"].median().rename("median_strike")
    tmp = sub.join(med, on=["TRADEDATE", "expiry_date"])
    tmp["dist_to_median_strike"] = (tmp["strike"] - tmp["median_strike"]).abs()
    atm_rows = tmp.sort_values(["TRADEDATE", "expiry_date", "dist_to_median_strike", "SECID"]).groupby(["TRADEDATE", "expiry_date"]).head(2)
    atm_proxy.append({
        "family": fam,
        "atm_proxy_rows": len(atm_rows),
        "atm_proxy_unique_secids": atm_rows["SECID"].nunique(),
        "atm_proxy_open_cov": atm_rows["OPEN"].notna().mean(),
        "atm_proxy_close_cov": atm_rows["CLOSE"].notna().mean(),
        "atm_proxy_value_cov": atm_rows["VALUE"].notna().mean(),
        "atm_proxy_volume_cov": atm_rows["VOLUME"].notna().mean(),
        "atm_proxy_settle_cov": atm_rows["SETTLEPRICE"].notna().mean(),
    })

display(pd.DataFrame(atm_proxy).sort_values("family"))


## Выбор семейства

### Что показала проверка
- `IM`: похоже на опционы **на сам IMOEX**, но не так явно на фьючерс; покрытие есть, но семейство более разреженное.
- `MM`: в названиях явно написано `на фьюч. контр. MXI-...`, то есть это как раз **опционы на фьючерс**; у него лучшее покрытие с 2024 года и больше всего строк.
- `MX`: тоже похоже на опционы на фьючерс (`MIX-...`), но семейство начинается позже, без покрытия 2024, и живёт в другом масштабе страйков.

### Практический итог
Если задача — **получить опционы на фьючерс на индекс IMOEX и их цены в наиболее полном виде**, то базовое рабочее семейство лучше выбрать **`MM`**.

Почему именно `MM`:
- это явно опционы **на фьючерсный контракт** (`MXI`);
- у него лучшее покрытие по 2024-2026;
- у него больше всего строк и SECID;
- `SETTLEPRICE` заполнен полностью, а trade coverage не хуже, чем у `IM` в среднем и лучше по полноте периода;
- страйки у `MM` находятся в понятном масштабе рядом с индексным уровнем, без смешения с `MX`-масштабом `165000+`.

### Что делаем дальше
- для основного анализа оставляем **`MM`**;
- `IM` можно держать как вспомогательное/сравнительное семейство;
- `MX` пока лучше не смешивать с ними в одном основном датасете.
